In [1]:
import os
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models, optimizers

In [2]:
IMG_SIZE = (48, 48)
BATCH_SIZE = 64
NUM_CLASSES = 7
EPOCHS = 15

train_dir = r"train"
test_dir  = r"test"

In [3]:
# 1) Генератори даних (аугментація + нормалізація)
# У статті згадується горизонтальне дзеркалення і аугментація для міноритарних класів.
# Тут задаємо типову аугментацію + horizontal_flip=True.
train_datagen = ImageDataGenerator(
    rescale=1.0/255.0,
    horizontal_flip=True,
    rotation_range=10,
    width_shift_range=0.10,
    height_shift_range=0.10,
    zoom_range=0.10
)

# Для тесту/валідації зазвичай тільки нормалізація
test_datagen = ImageDataGenerator(rescale=1.0/255.0)

train_gen = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    color_mode="grayscale",          # 48x48 grey-scale
    batch_size=BATCH_SIZE,
    class_mode="categorical",        # categorical_crossentropy
    shuffle=True
)

test_gen = test_datagen.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    color_mode="grayscale",
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

Found 28709 images belonging to 7 classes.
Found 7178 images belonging to 7 classes.


In [4]:
# 2) CNN архітектура: 4 Conv шари + 2 Dense (за описом)
model = models.Sequential([
    layers.Input(shape=(48, 48, 1)),

    # Conv block 1
    layers.Conv2D(64, (3, 3), padding="same"),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    # Conv block 2
    layers.Conv2D(128, (3, 3), padding="same"),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    # Conv block 3
    layers.Conv2D(512, (3, 3), padding="same"),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    # Conv block 4
    layers.Conv2D(512, (3, 3), padding="same"),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    # Flatten + 2 fully connected (Dense) layers
    layers.Flatten(),

    layers.Dense(256),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.25),

    layers.Dense(512),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.25),

    # Output: Dense + Softmax (7 класів)
    layers.Dense(NUM_CLASSES, activation="softmax")
])

In [5]:
# 3) Компіляція: Adam + categorical_crossentropy + accuracy (як в статті)
model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 48, 48, 64)          │             640 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 48, 48, 64)          │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ activation (Activation)              │ (None, 48, 48, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 24, 24, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 24, 24, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 24, 24, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_1                │ (None, 24, 24, 128)         │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ activation_1 (Activation)            │ (None, 24, 24, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 12, 12, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 12, 12, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 12, 12, 512)         │         590,336 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_2                │ (None, 12, 12, 512)         │           2,048 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ activation_2 (Activation)            │ (None, 12, 12, 512)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 6, 6, 512)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ (None, 6, 6, 512)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_3 (Conv2D)                    │ (None, 6, 6, 512)           │       2,359,808 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_3                │ (None, 6, 6, 512)           │           2,048 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ activation_3 (Activation)            │ (None, 6, 6, 512)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_3 (MaxPooling2D)       │ (None, 3, 3, 512)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼──────────────

 Total params: 4,347,655 (16.58 MB)

 Trainable params: 4,343,687 (16.57 MB)

 Non-trainable params: 3,968 (15.50 KB)

In [6]:
# 4) Навчання: 15 епох (як в статті)
history = model.fit(
    train_gen,
    epochs=EPOCHS,
    validation_data=test_gen
)


D:\ProgramFiles\Python3.10\lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/15
449/449 ━━━━━━━━━━━━━━━━━━━━ 280s 614ms/step - accuracy: 0.1978 - loss: 2.1280 - val_accuracy: 0.2473 - val_loss: 1.8970
Epoch 2/15
449/449 ━━━━━━━━━━━━━━━━━━━━ 257s 572ms/step - accuracy: 0.2481 - loss: 1.8965 - val_accuracy: 0.3171 - val_loss: 1.7213
Epoch 3/15
449/449 ━━━━━━━━━━━━━━━━━━━━ 245s 545ms/step - accuracy: 0.2676 - loss: 1.8481 - val_accuracy: 0.3314 - val_loss: 1.6879
Epoch 4/15
449/449 ━━━━━━━━━━━━━━━━━━━━ 233s 519ms/step - accuracy: 0.3037 - loss: 1.7719 - val_accuracy: 0.3534 - val_loss: 1.6983
Epoch 5/15
449/449 ━━━━━━━━━━━━━━━━━━━━ 235s 523ms/step - accuracy: 0.3281 - loss: 1.7145 - val_accuracy: 0.3540 - val_loss: 1.7543
Epoch 6/15
449/449 ━━━━━━━━━━━━━━━━━━━━ 233s 520ms/step - accuracy: 0.3510 - loss: 1.6615 - val_accuracy: 0.3945 - val_loss: 1.6287
Epoch 7/15
449/449 ━━━━━━━━━━━━━━━━━━━━ 232s 518ms/step - accuracy: 0.3747 - loss: 1.6156 - val_accuracy: 0.4161 - val_loss: 1.6050
Epoch 8/15
449/449 ━━━━━━━━━━━━━━━━━━━━ 233s 520ms/step - accuracy: 0.4005 -

In [7]:
# 5) Оцінювання
test_loss, test_acc = model.evaluate(test_gen, verbose=0)
print(f"Test accuracy: {test_acc:.4f}")


Test accuracy: 0.5371
